# Comparaison `Reassignment` / `Reassignment-v2` — Bat Full Spectrum Processing

Charge un WAV, lance les **deux pipelines indépendamment** et superpose leurs trajectoires sur **un spectrogramme Plotly Magma dont le fond est sélectionnable**.

- 🟢 **Vert (`Reassignment`)** : première version du suivi réassigné Flattop.
- 🔵 **Bleu (`Reassignment-v2`)** : nouvelle version du suivi réassigné Flattop.
- Les deux branches exécutent `adaptive_v4` pour proposer les chirps ; les courbes affichées viennent de leur **reconstruction**, pas seulement des pics du détecteur.
- Les commits Git utilisés figurent dans le titre et dans le récapitulatif.
- Des boutons permettent d'afficher **les deux branches**, **Reassignment seul**, **Reassignment-v2 seul**, ou **aucune trajectoire** sans relancer les calculs.
- Un autre sélecteur bascule entre **STFT Flattop**, **Reassignment Flattop 512/64**, **Reassignment du tracker** et sa **carte d'évidence locale**. Les trajectoires restent visibles sur tous les fonds.
- Les deux derniers modes sont calculés autour d'un **chirp V4 sélectionné**, avec les paramètres du tracker (`512/24`, ou `192/16` en dessous de 1,2 ms).
- Une **liste déroulante de chirps Reassignment-v2** permet de changer de candidat et de recalculer les vues locales, sans relancer la détection.

**Important :** ce notebook sélectionne explicitement `tracking_method="reassignment_flattop"` pour les **deux** branches. Elles sont exécutées dans deux processus Python distincts pour éviter les collisions d'import.

Exécuter les cellules dans l'ordre. Aucun changement n'est effectué sur GitHub.


In [ ]:
%pip -q install numpy scipy soundfile librosa plotly nbformat ipywidgets


## 1. Télécharger / actualiser les deux branches

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/dbrouste/bat_full_spectrum_processing.git"
ROOT = Path("/content") if Path("/content").is_dir() else Path.cwd()
BRANCHES = ("Reassignment", "Reassignment-v2")
REPO_DIRS = {branch: ROOT / f"bfsp_{branch.lower().replace('-', '_')}" for branch in BRANCHES}

for branch, directory in REPO_DIRS.items():
    if not (directory / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--single-branch", "--branch", branch,
             REPO_URL, str(directory)], check=True
        )
    else:
        # Préserve les éventuelles modifications locales ; échoue proprement
        # si un fast-forward n'est pas possible.
        subprocess.run(["git", "-C", str(directory), "pull", "--ff-only", "origin", branch], check=True)

COMMITS = {}
for branch, directory in REPO_DIRS.items():
    actual_branch = subprocess.check_output(
        ["git", "-C", str(directory), "branch", "--show-current"], text=True
    ).strip()
    if actual_branch != branch:
        raise RuntimeError(f"Branche incorrecte dans {directory} : {actual_branch} != {branch}")
    COMMITS[branch] = subprocess.check_output(
        ["git", "-C", str(directory), "rev-parse", "--short=12", "HEAD"], text=True
    ).strip()
    print(f"{branch:12} : {COMMITS[branch]}  ({directory})")


## 2. Choisir le WAV et les paramètres de visualisation

Laisser `WAV_PATH` vide pour envoyer un fichier depuis l'ordinateur ; ou renseigner son chemin Colab / Google Drive. `VIEW_START_S` et `VIEW_END_S` limitent **uniquement l'affichage**, les détecteurs travaillent sur tout le WAV. Pour zoomer encore davantage, utiliser ensuite Plotly.


In [ ]:
from pathlib import Path

# Exemple si Google Drive a été monté :
# WAV_PATH = "/content/drive/MyDrive/Bat/20261005_192441.wav"
WAV_PATH = ""

# Bornes d'affichage en secondes ; None = début / fin du WAV.
VIEW_START_S = None
VIEW_END_S = None
F_MIN_KHZ = 20.0
F_MAX_KHZ = 150.0
DB_FLOOR = -90.0       # dB relatifs au maximum du spectrogramme affiché
SPEC_NFFT = 1024       # spectrogramme de fond : STFT Flattop
MAX_PLOT_FRAMES = 2400 # limite la quantité de données envoyées à Plotly
REASSIGNMENT_NFFT = 512
REASSIGNMENT_HOP = 64  # 87,5 % de recouvrement : (512 - 64) / 512
REASSIGNMENT_SNR_MIN_DB = 4.0
REASSIGNMENT_SNR_FULL_DB = 16.0
SPECTRO_DEFAULT = "stft"  # "stft", "reassignment", "tracker_reassignment", "tracker_evidence"
MAX_CANDIDATES = None  # None : tous les candidats des deux branches
SHOW_DETECTOR_PEAKS = False
SHOW_REASSIGNMENT = True      # première branche, en vert
SHOW_REASSIGNMENT_V2 = True   # seconde branche, en bleu

if not WAV_PATH:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Veuillez sélectionner un seul fichier WAV.")
    WAV_PATH = next(iter(uploaded))

WAV_PATH = Path(WAV_PATH).expanduser().resolve()
if not WAV_PATH.is_file() or WAV_PATH.suffix.lower() != ".wav":
    raise FileNotFoundError(f"Fichier WAV introuvable : {WAV_PATH}")
print("WAV :", WAV_PATH)


## 3. Exécuter chaque branche isolément

Chaque exécution appelle le vrai `detect_candidates_snr_blobs(..., slope_filter_mode="adaptive_v4")`, puis `process_full_spectrum()` sur **sa propre branche**. Les deux reçoivent les bornes des blobs et l'option `reassignment_flattop`.

Les coordonnées temporelles de `process_full_spectrum()` sont relatives à un morceau de 15 ms ; elles sont reconverties en **temps absolu dans le WAV** avant de tracer.


In [ ]:
import json
import subprocess
import sys

_RUNNER = r"""
import json
import sys
import warnings
import numpy as np
import soundfile as sf

from bat_analysis.detection import detect_candidates_snr_blobs
from bat_analysis.modelling import process_full_spectrum

wav_path, branch, max_candidates_str = sys.argv[1:4]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    y, sr = sf.read(wav_path, always_2d=False)
y = np.asarray(y, dtype=np.float32)
if y.ndim == 2:
    y = y.mean(axis=1)
if y.size == 0:
    raise ValueError("WAV vide")

candidates = detect_candidates_snr_blobs(y, int(sr), slope_filter_mode="adaptive_v4")
candidates.sort(key=lambda c: float(c["time_mid"]))
max_candidates = int(max_candidates_str)
if max_candidates > 0:
    candidates = candidates[:max_candidates]

def finite_number(value):
    if value is None:
        return None
    try:
        value = float(value)
        return value if np.isfinite(value) else None
    except (TypeError, ValueError):
        return None

candidate_summaries = [
    {"id": index, **{
        key: finite_number(c.get(key))
        for key in ("time_mid", "duration", "peak_freq_hz",
                    "t_start", "t_end", "coarse_seed_time_s",
                    "broad_fm_seed_freq_hz", "coarse_seed_slope_hz_per_ms")
    }}
    for index, c in enumerate(candidates, start=1)
]

tracks = []
failures = []
for index, candidate in enumerate(candidates, start=1):
    mid = float(candidate["time_mid"])
    diagnostics = {}
    kwargs = dict(
        time_mid=mid,
        duration=float(candidate["duration"]),
        seed_freq_hz=float(candidate["peak_freq_hz"]),
        broad_fm_seed_freq_hz=candidate.get("broad_fm_seed_freq_hz"),
        broad_fm_seed_slope_hz_per_ms=candidate.get("coarse_seed_slope_hz_per_ms"),
        diagnostics=diagnostics,
    )
    if branch in ("Reassignment", "Reassignment-v2"):
        kwargs.update(
            tracking_method="reassignment_flattop",  # indispensable !
            chirp_start_s=float(candidate["t_start"]),
            chirp_end_s=float(candidate["t_end"]),
            broad_fm_seed_time_s=candidate.get("coarse_seed_time_s"),
        )
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            curve = process_full_spectrum(y, int(sr), **kwargs)
        if curve is None:
            failures.append({"id": index, "reason": str(diagnostics.get("stop_reason", "no_curve"))})
            continue
        curve = np.asarray(curve, dtype=float)
        if curve.ndim != 2 or curve.shape[1] < 2:
            failures.append({"id": index, "reason": "invalid_shape"})
            continue
        valid = np.isfinite(curve[:, :2]).all(axis=1) & (curve[:, 1] > 0)
        curve = curve[valid]
        if len(curve) < 2:
            failures.append({"id": index, "reason": "too_few_valid_points"})
            continue

        # Les DEUX modes rendent des instants relatifs au chunk de 15 ms.
        chunk_start_s = max(0, int((mid - 0.0075) * sr)) / sr
        t_ms = (curve[:, 0] + chunk_start_s) * 1000.0
        f_khz = curve[:, 1] / 1000.0
        order = np.argsort(t_ms)
        tracks.append({
            "id": index,
            "t_ms": t_ms[order].tolist(),
            "f_khz": f_khz[order].tolist(),
            "peak_t_ms": mid * 1000.0,
            "peak_f_khz": float(candidate["peak_freq_hz"]) / 1000.0,
            "tracker_n_fft": diagnostics.get("reassignment_n_fft"),
            "tracker_hop": diagnostics.get("reassignment_hop"),
        })
    except Exception as exc:
        failures.append({"id": index, "reason": f"{type(exc).__name__}: {exc}"[:220]})

print("__BFSP_RESULT__" + json.dumps({
    "branch": branch,
    "n_candidates": len(candidates),
    "candidates": candidate_summaries,
    "tracks": tracks,
    "failures": failures,
}))
"""

RESULTS = {}
for branch in BRANCHES:
    print(f"Analyse de {branch}...", flush=True)
    command = [sys.executable, "-c", _RUNNER, str(WAV_PATH), branch,
               str(MAX_CANDIDATES or 0)]
    result = subprocess.run(command, cwd=REPO_DIRS[branch], text=True,
                            capture_output=True, timeout=1200)
    lines = [line.removeprefix("__BFSP_RESULT__")
             for line in result.stdout.splitlines() if line.startswith("__BFSP_RESULT__")]
    if result.returncode or not lines:
        print(result.stdout[-3000:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"Échec de l'exécution de la branche {branch}")
    RESULTS[branch] = json.loads(lines[-1])
    data = RESULTS[branch]
    print(f"  {COMMITS[branch]} : {data['n_candidates']} candidats, "
          f"{len(data['tracks'])} courbes, {len(data['failures'])} échecs")
    if data["failures"]:
        print("  Exemples de rejet :", data["failures"][:3])


## 4. Choisir le chirp à inspecter dans le tracker

La liste déroulante, affichée **juste au-dessus du graphique**, contient **tous les candidats `adaptive_v4` de la branche `Reassignment-v2`**, y compris les candidats dont la trajectoire a été rejetée. Chaque choix indique le numéro, le temps et la fréquence du pic. Après avoir exécuté la cellule du graphique une première fois, le choix d'un autre chirp **recalcule automatiquement les deux vues locales** et affiche l'**Évidence élargie**, sans relancer les détections des branches.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass  # Fonctionne également dans Jupyter hors Colab.

TRACKER_CHIRP_ID = None  # Valeur initiale facultative, ex. 15
EVIDENCE_MARGIN_MS = 10.0  # Marge supplémentaire de chaque côté du chirp (>= 1 ms)

candidate_list = RESULTS["Reassignment-v2"]["candidates"]
successful_ids = [int(track["id"]) for track in RESULTS["Reassignment-v2"]["tracks"]]
initial_id = (int(TRACKER_CHIRP_ID) if TRACKER_CHIRP_ID is not None else
              (successful_ids[0] if successful_ids else
               (int(candidate_list[0]["id"]) if candidate_list else None)))
chirp_options = [
    (f"#{int(c['id']):03d} — {float(c['time_mid'])*1000:.3f} ms "
     f"/ {float(c['peak_freq_hz'])/1000:.1f} kHz", int(c["id"]))
    for c in candidate_list
]
if initial_id is not None and initial_id not in [value for _, value in chirp_options]:
    raise ValueError(f"TRACKER_CHIRP_ID={initial_id} absent des candidats Reassignment-v2")
CHIRP_SELECTOR = widgets.Dropdown(
    options=chirp_options or [("Aucun candidat", None)],
    value=initial_id, description="Chirp :", disabled=not bool(chirp_options),
    style={"description_width": "initial"}, layout=widgets.Layout(width="560px"),
)
print(f"{len(candidate_list)} candidats Reassignment-v2, dont {len(successful_ids)} avec trajectoire.")


## 5. Spectrogramme Magma et trajectoires superposées

Déplacer, zoomer, survoler les trajectoires. Deux lignes de boutons en haut permettent de sélectionner **le fond spectrogramme** et **les trajectoires visibles**, indépendamment et sans relancer la détection. La légende permet également de masquer ou d'isoler une branche.

La STFT conserve sa résolution paramétrable (`SPEC_NFFT=1024` par défaut). Le Reassignment global est calculé avec `n_fft=512`, `hop=64` et un masque SNR progressif de 4 à 16 dB. Le **Tracker Reassignment-v2** conserve exactement sa fenêtre originale (bbox du chirp ± 1 ms). La vue **Évidence élargie** recalcule la carte sur une région plus grande (`EVIDENCE_MARGIN_MS`, 10 ms de part et d'autre par défaut), avec la résolution et les pondérations du tracker (512/24 ou 192/16, percentile de bruit 20). Elle reste calculée **avant** l'optimisation dynamique.

**Attention :** la carte d'évidence élargie est une représentation de diagnostic. Comme le percentile de bruit est recalculé sur une fenêtre plus grande, ses valeurs peuvent différer de la carte strictement utilisée par le tracker. Les trajectoires affichées ne sont pas recalculées.

Chaque spectrogramme est normalisé séparément pour l'affichage ; la couleur indique un **niveau relatif**, pas une égalité d'amplitude entre traitements. La carte d'évidence utilise son échelle propre de 0 à 1. Les modes locaux partagent exactement le **même cadrage temporel** (celui de l'évidence élargie), même si les données du Tracker Reassignment-v2 ne sont calculées que dans sa fenêtre originale. Pour changer de chirp, utiliser la **liste déroulante de la cellule précédente** : la figure se met à jour automatiquement et revient sur l'évidence élargie.

**Lecture des nombres :** le nombre total de courbes calculées peut être supérieur au nombre affiché, notamment si `VIEW_START_S` / `VIEW_END_S` limitent la période, si la plage fréquentielle masque des courbes ou si les deux trajectoires se superposent presque exactement. Les traits verts sont plus épais et les bleus sont en pointillés pour rendre les superpositions visibles.


In [ ]:
import numpy as np
import soundfile as sf
import importlib.util
from scipy import signal
import plotly.graph_objects as go

y, sr = sf.read(str(WAV_PATH), always_2d=False)
if y.ndim == 2:
    y = y.mean(axis=1)
total_s = len(y) / sr
start_s = max(0.0, float(VIEW_START_S)) if VIEW_START_S is not None else 0.0
end_s = min(total_s, float(VIEW_END_S)) if VIEW_END_S is not None else total_s
if not 0 <= start_s < end_s <= total_s:
    raise ValueError("VIEW_START_S / VIEW_END_S invalides")

first_sample = int(start_s * sr)
last_sample = min(len(y), int(np.ceil(end_s * sr)))
y_view = np.asarray(y[first_sample:last_sample], dtype=np.float32)
nfft = min(SPEC_NFFT, len(y_view))
if nfft < 32:
    raise ValueError("Fenêtre trop courte pour calculer le spectrogramme")
# Un hop adaptatif rend la figure fluide même avec un long WAV.
hop = max(32, min(nfft - 1, int(np.ceil(len(y_view) / MAX_PLOT_FRAMES))))
frequencies, times, stft = signal.stft(
    y_view, fs=sr, window="flattop", nperseg=nfft,
    noverlap=nfft - hop, boundary=None, padded=False,
)
time_ms = 1000.0 * (times + first_sample / sr)
freq_khz = frequencies / 1000.0
band = (freq_khz >= F_MIN_KHZ) & (freq_khz <= min(F_MAX_KHZ, sr / 2000.0))
if not band.any():
    raise ValueError("La plage de fréquences demandée est hors Nyquist")

power = np.abs(stft[band]) ** 2
reference = max(float(np.max(power)), 1e-30)
db = 10 * np.log10(np.maximum(power, 1e-30) / reference)
step = max(1, int(np.ceil(db.shape[1] / MAX_PLOT_FRAMES)))
time_ms = time_ms[::step]
db = db[:, ::step]

SPECTRO_MODES = ("stft", "reassignment", "tracker_reassignment", "tracker_evidence")
if SPECTRO_DEFAULT not in SPECTRO_MODES:
    raise ValueError(f"SPECTRO_DEFAULT doit être dans {SPECTRO_MODES}")
if REASSIGNMENT_NFFT != 512 or REASSIGNMENT_HOP != 64:
    raise ValueError("Cette comparaison est configurée pour 512 points et hop=64 (87,5 % de recouvrement)")
if len(y_view) < REASSIGNMENT_NFFT:
    raise ValueError("Fenêtre d'affichage trop courte : au moins 512 échantillons requis")

# Charger la carte réassignée depuis la branche Reassignment-v2.
# Ne pas importer bat_analysis globalement : la détection des deux branches
# est exécutée dans des processus séparés pour éviter les collisions.
module_file = REPO_DIRS["Reassignment-v2"] / "bat_analysis" / "reassignment.py"
spec = importlib.util.spec_from_file_location("_bfsp_reassignment_view", module_file)
reassignment_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reassignment_module)

r = reassignment_module.reassigned_spectrogram(
    y_view, int(sr), n_fft=REASSIGNMENT_NFFT, hop=REASSIGNMENT_HOP,
    fmin=F_MIN_KHZ * 1000.0, fmax=min(F_MAX_KHZ * 1000.0, sr / 2),
    snr_min_db=REASSIGNMENT_SNR_MIN_DB,
    snr_full_db=REASSIGNMENT_SNR_FULL_DB,
)
reassigned_time_ms = 1000.0 * (r["times_s"] + first_sample / sr)
reassigned_freq_khz = r["freqs_hz"] / 1000.0
reassigned_power = r["reassigned_power"]

# Le reassignment est clairsemé : conserver le maximum de chaque paquet de
# colonnes, plutôt que sous-échantillonner les frames (qui perdrait les ridges).
if reassigned_power.shape[1] > MAX_PLOT_FRAMES:
    stride = int(np.ceil(reassigned_power.shape[1] / MAX_PLOT_FRAMES))
    starts = np.arange(0, reassigned_power.shape[1], stride)
    reassigned_power = np.maximum.reduceat(reassigned_power, starts, axis=1)
    reassigned_time_ms = reassigned_time_ms[np.minimum(starts + stride // 2,
                                                        len(reassigned_time_ms) - 1)]

ref_reassigned = max(float(reassigned_power.max()), 1e-30)
reassigned_db = 10.0 * np.log10(np.maximum(reassigned_power, 1e-30) / ref_reassigned)


from IPython.display import clear_output, display
import traceback

def _plot_selected_chirp(chirp_id, mode_override=None):
    global fig  # Réutilisé par la cellule d'export HTML.
    active_mode = SPECTRO_DEFAULT if mode_override is None else mode_override
    # Reproduire la construction de la carte interne à track_reassigned_ridge.
    # L'algorithme travaille candidat par candidat, sur son bbox élargi de 1 ms.
    from scipy.ndimage import gaussian_filter

    reassignment_candidates = RESULTS["Reassignment-v2"]["candidates"]
    LOCAL_AVAILABLE = bool(reassignment_candidates)
    selected_id = None
    if LOCAL_AVAILABLE:
        successful_ids = [int(t["id"]) for t in RESULTS["Reassignment-v2"]["tracks"]]
        selected_id = (int(chirp_id) if chirp_id is not None
                       else (successful_ids[0] if successful_ids else 1))
        if not 1 <= selected_id <= len(reassignment_candidates):
            raise ValueError(f"Le chirp sélectionné doit être entre 1 et {len(reassignment_candidates)}")
        candidate = reassignment_candidates[selected_id - 1]
        duration = float(candidate["duration"])
        selected_track = next((t for t in RESULTS["Reassignment-v2"]["tracks"]
                               if int(t["id"]) == selected_id), None)
        # Priorité aux paramètres effectivement consignés par le tracker.
        tracker_n_fft = (int(selected_track["tracker_n_fft"])
                         if selected_track and selected_track.get("tracker_n_fft") is not None
                         else (192 if duration < 0.0012 else 512))
        tracker_hop = (int(selected_track["tracker_hop"])
                       if selected_track and selected_track.get("tracker_hop") is not None
                       else (16 if tracker_n_fft <= 256 else 24))
        if candidate["t_start"] is None or candidate["t_end"] is None:
            raise ValueError("Bornes du candidat absentes")
        # Exactement les limites choisies dans reassignment_tracking.py quand
        # process_full_spectrum fournit chirp_start_s et chirp_end_s.
        tracker_start_s = max(0.0, float(candidate["t_start"]) - 0.001)
        tracker_stop_s = min(total_s, float(candidate["t_end"]) + 0.001)
        tracker_first = max(0, int(tracker_start_s * sr) - tracker_n_fft)
        tracker_last = min(len(y), int(tracker_stop_s * sr) + tracker_n_fft)
        if tracker_last - tracker_first < tracker_n_fft:
            raise ValueError("Fenêtre du candidat trop courte pour le tracker")

        tracker_r = reassignment_module.reassigned_spectrogram(
            y[tracker_first:tracker_last], int(sr),
            n_fft=tracker_n_fft, hop=tracker_hop,
            fmin=20000.0, fmax=min(150000.0, sr / 2),
            noise_percentile=20.0, snr_min_db=4.0, snr_full_db=16.0,
        )
        tracker_all_times = tracker_r["times_s"] + tracker_first / sr
        tracker_keep = ((tracker_all_times >= tracker_start_s)
                        & (tracker_all_times <= tracker_stop_s))
        tracker_times_ms = 1000.0 * tracker_all_times[tracker_keep]
        tracker_freq_khz = tracker_r["freqs_hz"] / 1000.0
        tracker_regular = tracker_r["stft_power"][:, tracker_keep]
        tracker_reassigned = tracker_r["reassigned_power"][:, tracker_keep]
        tracker_snr = tracker_r["snr_db"][:, tracker_keep]

        if len(tracker_times_ms) < 4 or not tracker_reassigned.any():
            raise ValueError("Candidat sans spectrogramme réassigné exploitable ; "
                             "essayer un autre chirp dans la liste déroulante")

        # Même formule que dans reassignment_tracking.py, réutilisée pour
        # la fenêtre réelle du tracker et pour le diagnostic élargi.
        def compute_tracker_evidence(regular, reassigned, snr):
            focused = gaussian_filter(reassigned, (1.2, 0.7), mode="constant")
            broad = gaussian_filter(regular, (1.0, 0.5), mode="constant")
            focused_db = 10 * np.log10(np.maximum(focused, 1e-30))
            broad_db = 10 * np.log10(np.maximum(broad, 1e-30))
            floor = np.percentile(broad_db, 20, axis=1)
            evidence = np.clip(
                0.65 * np.maximum(0, focused_db - floor[:, None] - 2)
                + 0.35 * np.maximum(0, broad_db - floor[:, None] - 3),
                0, 32,
            ) / 32
            evidence[snr < 2] *= 0.3
            return evidence

        tracker_evidence = compute_tracker_evidence(
            tracker_regular, tracker_reassigned, tracker_snr)

        tracker_ref = max(float(tracker_reassigned.max()), 1e-30)
        tracker_reassigned_db = 10 * np.log10(
            np.maximum(tracker_reassigned, 1e-30) / tracker_ref)
        LOCAL_RANGE_MS = [tracker_start_s * 1000.0, tracker_stop_s * 1000.0]

        # Évidence contextualisée sur une fenêtre réellement plus grande.
        margin_s = float(EVIDENCE_MARGIN_MS) / 1000.0
        if not np.isfinite(margin_s) or margin_s < 0.001:
            raise ValueError("EVIDENCE_MARGIN_MS doit être >= 1 ms et fini")
        evidence_start_s = max(0.0, float(candidate["t_start"]) - margin_s)
        evidence_stop_s = min(total_s, float(candidate["t_end"]) + margin_s)
        evidence_first = max(0, int(evidence_start_s * sr) - tracker_n_fft)
        evidence_last = min(len(y), int(evidence_stop_s * sr) + tracker_n_fft)
        evidence_r = reassignment_module.reassigned_spectrogram(
            y[evidence_first:evidence_last], int(sr),
            n_fft=tracker_n_fft, hop=tracker_hop,
            fmin=20000.0, fmax=min(150000.0, sr / 2),
            noise_percentile=20.0, snr_min_db=4.0, snr_full_db=16.0,
        )
        evidence_all_times = evidence_r["times_s"] + evidence_first / sr
        evidence_keep = ((evidence_all_times >= evidence_start_s)
                         & (evidence_all_times <= evidence_stop_s))
        evidence_times_ms = 1000.0 * evidence_all_times[evidence_keep]
        evidence_freq_khz = evidence_r["freqs_hz"] / 1000.0
        if evidence_times_ms.size < 4:
            raise ValueError("Fenêtre d'évidence élargie trop courte")
        extended_evidence = compute_tracker_evidence(
            evidence_r["stft_power"][:, evidence_keep],
            evidence_r["reassigned_power"][:, evidence_keep],
            evidence_r["snr_db"][:, evidence_keep],
        )
        EVIDENCE_RANGE_MS = [evidence_start_s * 1000.0, evidence_stop_s * 1000.0]
        print(f"Chirp Reassignment-v2 #{selected_id}, pic "
              f"{candidate['time_mid'] * 1000:.3f} ms / "
              f"{candidate['peak_freq_hz'] / 1000:.1f} kHz : "
              f"tracker Flattop {tracker_n_fft}/{tracker_hop}, "
              f"fenêtre {LOCAL_RANGE_MS[0]:.3f}-{LOCAL_RANGE_MS[1]:.3f} ms")
        print(f"Évidence élargie : {EVIDENCE_RANGE_MS[0]:.3f}–"
              f"{EVIDENCE_RANGE_MS[1]:.3f} ms (±{EVIDENCE_MARGIN_MS:g} ms)")
    else:
        if active_mode in ("tracker_reassignment", "tracker_evidence"):
            raise ValueError("Aucun candidat V4 dans ce WAV : modes locaux indisponibles")
        print("Aucun candidat Reassignment-v2 : vues locales du tracker indisponibles")

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        x=time_ms, y=freq_khz[band], z=np.clip(db, DB_FLOOR, 0),
        colorscale="Magma", zmin=DB_FLOOR, zmax=0,
        colorbar=dict(title="dB rel.", thickness=14),
        hovertemplate="t = %{x:.3f} ms<br>f = %{y:.2f} kHz<br>P = %{z:.1f} dB<extra></extra>",
        name="STFT Flattop",
        showscale=True,
        visible=active_mode == "stft",
    ))
    fig.add_trace(go.Heatmap(
        x=reassigned_time_ms, y=reassigned_freq_khz,
        z=np.clip(reassigned_db, DB_FLOOR, 0),
        colorscale="Magma", zmin=DB_FLOOR, zmax=0,
        colorbar=dict(title="dB rel.", thickness=14),
        hovertemplate="Reassignment<br>t = %{x:.3f} ms<br>f = %{y:.2f} kHz<br>P = %{z:.1f} dB<extra></extra>",
        name="Reassignment Flattop 512/64",
        showscale=True,
        visible=active_mode == "reassignment",
    ))

    if LOCAL_AVAILABLE:
        fig.add_trace(go.Heatmap(
            x=tracker_times_ms, y=tracker_freq_khz,
            z=np.clip(tracker_reassigned_db, DB_FLOOR, 0),
            colorscale="Magma", zmin=DB_FLOOR, zmax=0,
            colorbar=dict(title="dB rel.", thickness=14),
            hovertemplate="Reassignment tracker<br>t = %{x:.3f} ms<br>f = %{y:.2f} kHz<br>P = %{z:.1f} dB<extra></extra>",
            name=f"Reassignment-v2 tracker #{selected_id} {tracker_n_fft}/{tracker_hop}",
            showscale=True, visible=active_mode == "tracker_reassignment",
        ))
        fig.add_trace(go.Heatmap(
            x=evidence_times_ms, y=evidence_freq_khz, z=extended_evidence,
            colorscale="Magma", zmin=0, zmax=1,
            colorbar=dict(title="Évidence", thickness=14),
            hovertemplate="Évidence élargie<br>t = %{x:.3f} ms<br>f = %{y:.2f} kHz<br>E = %{z:.3f}<extra></extra>",
            name=f"Évidence élargie #{selected_id}",
            showscale=True, visible=active_mode == "tracker_evidence",
        ))

    COLORS = {"Reassignment": "#18e16f", "Reassignment-v2": "#35a1ff"}
    INITIAL_VISIBILITY = {"Reassignment": bool(SHOW_REASSIGNMENT),
                          "Reassignment-v2": bool(SHOW_REASSIGNMENT_V2)}
    N_BACKGROUNDS = len(fig.data)
    TRACE_BRANCHES = [None] * N_BACKGROUNDS  # Une entrée par spectrogramme de fond.
    VISIBLE_COUNTS = {}
    for branch in BRANCHES:
        tracks = RESULTS[branch]["tracks"]
        visible_index = 0
        local_index = 0
        for track in tracks:
            t = np.asarray(track["t_ms"], dtype=float)
            f = np.asarray(track["f_khz"], dtype=float)
            within_global = ((t >= start_s * 1000.0) & (t <= end_s * 1000.0)
                             & (f >= F_MIN_KHZ) & (f <= min(F_MAX_KHZ, sr / 2000.0)))
            within_local = ((t >= LOCAL_RANGE_MS[0]) & (t <= LOCAL_RANGE_MS[1])
                            & (f >= F_MIN_KHZ) & (f <= min(F_MAX_KHZ, sr / 2000.0))) if LOCAL_AVAILABLE else np.zeros_like(within_global)
            within_evidence = ((t >= EVIDENCE_RANGE_MS[0]) & (t <= EVIDENCE_RANGE_MS[1])
                               & (f >= F_MIN_KHZ) & (f <= min(F_MAX_KHZ, sr / 2000.0))) if LOCAL_AVAILABLE else np.zeros_like(within_global)
            if np.count_nonzero(within_local) >= 2:
                local_index += 1
            within = within_global | within_local | within_evidence
            if np.count_nonzero(within) < 2:
                continue
            # Les NaN font des ruptures lorsque la trajectoire sort de l'affichage.
            fig.add_trace(go.Scatter(
                x=np.where(within, t, np.nan),
                y=np.where(within, f, np.nan),
                mode="lines", line=dict(color=COLORS[branch],
                                        width=3.6 if branch == "Reassignment" else 2.0,
                                        dash="solid" if branch == "Reassignment" else "dash"),
                visible=INITIAL_VISIBILITY[branch],
                name=branch,
                legendgroup=branch, showlegend=(visible_index == 0),
                hovertemplate=(f"{branch}, chirp {track['id']}<br>"
                               "t = %{x:.3f} ms<br>f = %{y:.2f} kHz<extra></extra>"),
            ))
            TRACE_BRANCHES.append(branch)
            visible_index += 1

        VISIBLE_COUNTS[branch] = (sum(
            np.count_nonzero(
                (np.asarray(t["t_ms"]) >= start_s * 1000.0)
                & (np.asarray(t["t_ms"]) <= end_s * 1000.0)
                & (np.asarray(t["f_khz"]) >= F_MIN_KHZ)
                & (np.asarray(t["f_khz"]) <= min(F_MAX_KHZ, sr / 2000.0))
            ) >= 2 for t in tracks))
        if LOCAL_AVAILABLE:
            print(f"{branch} : {local_index} trajectoires dans la fenêtre du chirp #{selected_id}")
        if SHOW_DETECTOR_PEAKS:
            fig.add_trace(go.Scatter(
                x=[t["peak_t_ms"] for t in tracks],
                y=[t["peak_f_khz"] for t in tracks],
                mode="markers", marker=dict(color=COLORS[branch], size=7, symbol="x"),
                visible=INITIAL_VISIBILITY[branch],
                name=f"Pics {branch}", legendgroup=f"peaks_{branch}",
                hovertemplate="Pic %{x:.3f} ms / %{y:.2f} kHz<extra></extra>",
            ))
            TRACE_BRANCHES.append(branch)

    # Les menus Pilotent chacun leurs propres traces pour que changer de fond ne
    # modifie jamais les annotations, et vice versa.
    def _visibility(show_reassignment, show_reassignment_v2):
        return [show_reassignment if branch == "Reassignment" else show_reassignment_v2
                for branch in TRACE_BRANCHES[N_BACKGROUNDS:]]

    view_buttons = [
        dict(label=label, method="restyle",
             args=[{"visible": _visibility(reassignment, reassignment_v2)},
                   list(range(N_BACKGROUNDS, len(TRACE_BRANCHES)))])
        for label, reassignment, reassignment_v2 in [
            ("Les deux", True, True),
            ("Reassignment seul", True, False),
            ("Reassignment-v2 seul", False, True),
            ("Aucune", False, False),
        ]
    ]
    mode_labels = [("STFT Flattop", "stft"), ("Reassignment 512/64", "reassignment")]
    if LOCAL_AVAILABLE:
        mode_labels += [
            (f"Reassignment suivi {tracker_n_fft}/{tracker_hop}", "tracker_reassignment"),
            (f"Évidence élargie #{selected_id}", "tracker_evidence"),
        ]
    spectro_buttons = []
    for label, mode in mode_labels:
        if mode in ("tracker_reassignment", "tracker_evidence"):
            view_range = EVIDENCE_RANGE_MS
        else:
            view_range = [start_s * 1000.0, end_s * 1000.0]
        layout_update = {"xaxis.range": view_range, "xaxis.autorange": False}
        spectro_buttons.append(dict(
            label=label, method="update",
            args=[{"visible": [mode == value for value in SPECTRO_MODES[:N_BACKGROUNDS]]},
                  layout_update, list(range(N_BACKGROUNDS))],
        ))

    initial_branch_button = {
        (True, True): 0, (True, False): 1,
        (False, True): 2, (False, False): 3,
    }[(bool(SHOW_REASSIGNMENT), bool(SHOW_REASSIGNMENT_V2))]

    fig.update_layout(
        template="plotly_dark",
        title=(f"{WAV_PATH.name} · "
               f"Reassignment {COMMITS['Reassignment']} (vert) / "
               f"Reassignment-v2 {COMMITS['Reassignment-v2']} (bleu)"),
        xaxis_title="Temps dans le WAV (ms)",
        yaxis_title="Fréquence (kHz)",
        xaxis=dict(range=(
            EVIDENCE_RANGE_MS if LOCAL_AVAILABLE and active_mode in ("tracker_reassignment", "tracker_evidence")
            else [start_s * 1000, end_s * 1000]
        ), showgrid=False, autorange=False),
        yaxis=dict(range=[F_MIN_KHZ, min(F_MAX_KHZ, sr / 2000)], showgrid=False),
        legend=dict(groupclick="togglegroup", bgcolor="rgba(0,0,0,0.35)"),
        updatemenus=[
            dict(type="buttons", direction="right", buttons=spectro_buttons,
                 x=0.17, xanchor="left", y=1.22, yanchor="bottom",
                 active=[mode for _, mode in mode_labels].index(active_mode),
                 showactive=True, bgcolor="rgba(70,70,80,0.90)"),
            dict(type="buttons", direction="right", buttons=view_buttons,
                 x=0.17, xanchor="left", y=1.11, yanchor="bottom",
                 active=initial_branch_button,
                 showactive=True, bgcolor="rgba(70,70,80,0.90)"),
        ],
        annotations=[
            dict(text="Fond :", x=0.01, y=1.26, xref="paper", yref="paper",
                 showarrow=False, xanchor="left"),
            dict(text="Trajectoires :", x=0.01, y=1.15, xref="paper", yref="paper",
                 showarrow=False, xanchor="left"),
        ],
        height=760,
        margin=dict(l=70, r=40, t=200, b=60),
        hovermode="closest",
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})

    print("Fonds disponibles : " + ", ".join(label for label, _ in mode_labels))
    print("Révisions comparées :")
    for branch in BRANCHES:
        data = RESULTS[branch]
        print(f"  {branch:12} {COMMITS[branch]} | "
              f"{data['n_candidates']} candidats, {len(data['tracks'])} courbes calculées, "
              f"{VISIBLE_COUNTS[branch]} dans la fenêtre affichée")
    if any(VISIBLE_COUNTS[b] < len(RESULTS[b]["tracks"]) for b in BRANCHES):
        print("Une partie des courbes est hors fenêtre : vérifier VIEW_START_S / "
              "VIEW_END_S et F_MIN_KHZ / F_MAX_KHZ.")


_CHIRP_PLOT_OUTPUT = widgets.Output()

def _refresh_chirp_plot(change=None):
    with _CHIRP_PLOT_OUTPUT:
        clear_output(wait=True)
        try:
            # Après un changement, montrer directement l'évidence du chirp choisi.
            mode = 'tracker_evidence' if change is not None else None
            _plot_selected_chirp(CHIRP_SELECTOR.value, mode_override=mode)
        except Exception:
            traceback.print_exc()

# Ne pas empiler des callbacks lorsque cette cellule est exécutée à nouveau.
if '_CHIRP_BOUND_SELECTOR' in globals() and '_CHIRP_CHANGE_HANDLER' in globals():
    _CHIRP_BOUND_SELECTOR.unobserve(_CHIRP_CHANGE_HANDLER, names='value')
_CHIRP_CHANGE_HANDLER = _refresh_chirp_plot
_CHIRP_BOUND_SELECTOR = CHIRP_SELECTOR
CHIRP_SELECTOR.observe(_CHIRP_CHANGE_HANDLER, names='value')
display(CHIRP_SELECTOR, _CHIRP_PLOT_OUTPUT)
_refresh_chirp_plot()


## 6. Export HTML interactif (facultatif)

Le fichier HTML conserve le spectrogramme, les couleurs, les zooms Plotly et les trajectoires. Il est indépendant de Colab une fois téléchargé.


In [ ]:
HTML_PATH = ROOT / f"comparaison_reassignment_v2_{WAV_PATH.stem}.html"
fig.write_html(str(HTML_PATH), include_plotlyjs=True)
print("HTML enregistré :", HTML_PATH)

# Pour le récupérer sur votre ordinateur :
# from google.colab import files
# files.download(str(HTML_PATH))
